In [18]:

import re
import itertools
import textwrap
import networkx as nx
import matplotlib.pyplot as plt


# ============================================================
# 1. TOKENIZER
# ============================================================

TOKEN_PATTERN = re.compile(
    r"\s*(forall|exists|implies|iff|and|or|not|"
    r"[A-Za-z_][A-Za-z_0-9]*|[(),])"
)


def tokenize(expression):
    tokens = []
    position = 0

    while position < len(expression):

        if expression[position:].strip() == "":
            break

        match = TOKEN_PATTERN.match(expression, position)

        if not match:
            raise ValueError(
                "Invalid input near: " + expression[position:]
            )

        tokens.append(match.group(1))
        position = match.end()

    return tokens


# ============================================================
# 2. FOL PARSER
# ============================================================

class Parser:

    def __init__(self, tokens):
        self.tokens = tokens
        self.position = 0

    def peek(self):

        if self.position < len(self.tokens):
            return self.tokens[self.position]

        return None

    def consume(self, expected=None):

        token = self.peek()

        if token is None:
            raise ValueError("Unexpected end of expression")

        if expected is not None and token != expected:
            raise ValueError(
                f"Expected {expected}, found {token}"
            )

        self.position += 1

        return token

    def parse(self):

        result = self.parse_iff()

        if self.peek() is not None:
            raise ValueError(
                "Unexpected token: " + str(self.peek())
            )

        return result

    def parse_iff(self):

        left = self.parse_implies()

        while self.peek() == "iff":

            self.consume()

            right = self.parse_implies()

            left = ("iff", left, right)

        return left

    def parse_implies(self):

        left = self.parse_or()

        if self.peek() == "implies":

            self.consume()

            right = self.parse_implies()

            return ("implies", left, right)

        return left

    def parse_or(self):

        left = self.parse_and()

        while self.peek() == "or":

            self.consume()

            right = self.parse_and()

            left = ("or", left, right)

        return left

    def parse_and(self):

        left = self.parse_unary()

        while self.peek() == "and":

            self.consume()

            right = self.parse_unary()

            left = ("and", left, right)

        return left

    def parse_unary(self):

        token = self.peek()

        if token == "not":

            self.consume()

            return ("not", self.parse_unary())

        if token == "forall":

            self.consume()

            variable = self.consume()

            body = self.parse_unary()

            return ("forall", variable, body)

        if token == "exists":

            self.consume()

            variable = self.consume()

            body = self.parse_unary()

            return ("exists", variable, body)

        if token == "(":

            self.consume("(")

            result = self.parse_iff()

            self.consume(")")

            return result

        return self.parse_predicate()

    def parse_predicate(self):

        name = self.consume()

        if not re.fullmatch(r"[A-Za-z_][A-Za-z_0-9]*", name):
            raise ValueError("Invalid predicate name")

        self.consume("(")

        arguments = []

        if self.peek() != ")":

            arguments.append(self.consume())

            while self.peek() == ",":

                self.consume(",")

                arguments.append(self.consume())

        self.consume(")")

        return ("pred", name, arguments)


def parse_expression(expression):

    return Parser(tokenize(expression)).parse()


# ============================================================
# 3. DISPLAY FOL WITH MATHEMATICAL SYMBOLS
# ============================================================

def show(node):

    kind = node[0]

    if kind == "pred":

        return (
            node[1]
            + "("
            + ", ".join(node[2])
            + ")"
        )

    if kind == "not":
        return "¬" + show(node[1])

    if kind == "and":

        return (
            "(" + show(node[1])
            + " ∧ "
            + show(node[2]) + ")"
        )

    if kind == "or":

        return (
            "(" + show(node[1])
            + " ∨ "
            + show(node[2]) + ")"
        )

    if kind == "implies":

        return (
            "(" + show(node[1])
            + " → "
            + show(node[2]) + ")"
        )

    if kind == "iff":

        return (
            "(" + show(node[1])
            + " ↔ "
            + show(node[2]) + ")"
        )

    if kind == "forall":

        return "∀" + node[1] + " (" + show(node[2]) + ")"

    if kind == "exists":

        return "∃" + node[1] + " (" + show(node[2]) + ")"

    return str(node)


# ============================================================
# 4. ELIMINATE IMPLICATIONS
# ============================================================

def eliminate_implications(node):

    kind = node[0]

    if kind == "pred":
        return node

    if kind == "not":

        return (
            "not",
            eliminate_implications(node[1])
        )

    if kind in ("and", "or"):

        return (
            kind,
            eliminate_implications(node[1]),
            eliminate_implications(node[2])
        )

    if kind == "implies":

        # A -> B = not A OR B

        return (
            "or",
            ("not", eliminate_implications(node[1])),
            eliminate_implications(node[2])
        )

    if kind == "iff":

        # A <-> B =
        # (not A OR B) AND (not B OR A)

        a = eliminate_implications(node[1])
        b = eliminate_implications(node[2])

        return (
            "and",
            ("or", ("not", a), b),
            ("or", ("not", b), a)
        )

    if kind in ("forall", "exists"):

        return (
            kind,
            node[1],
            eliminate_implications(node[2])
        )

    return node


# ============================================================
# 5. NEGATION NORMAL FORM
# ============================================================

def make_nnf(node, negated=False):

    kind = node[0]

    if kind == "pred":

        if negated:
            return ("not", node)

        return node

    if kind == "not":

        return make_nnf(node[1], not negated)

    if kind in ("forall", "exists"):

        new_kind = kind

        if negated:

            new_kind = (
                "exists" if kind == "forall"
                else "forall"
            )

        return (
            new_kind,
            node[1],
            make_nnf(node[2], negated)
        )

    if kind in ("and", "or"):

        new_kind = kind

        if negated:

            new_kind = (
                "or" if kind == "and"
                else "and"
            )

        return (
            new_kind,
            make_nnf(node[1], negated),
            make_nnf(node[2], negated)
        )

    return node


# ============================================================
# 6. VARIABLE SUBSTITUTION
# ============================================================

def substitute(node, variable, replacement):

    kind = node[0]

    if kind == "pred":

        args = [
            replacement if arg == variable else arg
            for arg in node[2]
        ]

        return ("pred", node[1], args)

    if kind == "not":

        return (
            "not",
            substitute(node[1], variable, replacement)
        )

    if kind in ("and", "or"):

        return (
            kind,
            substitute(node[1], variable, replacement),
            substitute(node[2], variable, replacement)
        )

    if kind in ("forall", "exists"):

        if node[1] == variable:
            return node

        return (
            kind,
            node[1],
            substitute(node[2], variable, replacement)
        )

    return node


# ============================================================
# 7. SKOLEMIZATION
# ============================================================

skolem_counter = 0


def skolemize(node, universal_variables=None):

    global skolem_counter

    if universal_variables is None:
        universal_variables = []

    kind = node[0]

    if kind == "pred":
        return node

    if kind == "not":

        return (
            "not",
            skolemize(node[1], universal_variables)
        )

    if kind in ("and", "or"):

        return (
            kind,
            skolemize(node[1], universal_variables),
            skolemize(node[2], universal_variables)
        )

    if kind == "forall":

        new_variables = universal_variables + [node[1]]

        return skolemize(node[2], new_variables)

    if kind == "exists":

        skolem_counter += 1

        if universal_variables:

            replacement = (
                "SK" + str(skolem_counter)
                + "(" + ",".join(universal_variables) + ")"
            )

        else:

            replacement = "SK" + str(skolem_counter)

        body = substitute(
            node[2],
            node[1],
            replacement
        )

        return skolemize(body, universal_variables)

    return node


# ============================================================
# 8. DISTRIBUTE OR OVER AND
# ============================================================

def distribute_or(a, b):

    if a[0] == "and":

        return (
            "and",
            distribute_or(a[1], b),
            distribute_or(a[2], b)
        )

    if b[0] == "and":

        return (
            "and",
            distribute_or(a, b[1]),
            distribute_or(a, b[2])
        )

    return ("or", a, b)


def convert_to_cnf(node):

    kind = node[0]

    if kind in ("pred", "not"):
        return node

    if kind == "and":

        return (
            "and",
            convert_to_cnf(node[1]),
            convert_to_cnf(node[2])
        )

    if kind == "or":

        left = convert_to_cnf(node[1])
        right = convert_to_cnf(node[2])

        return distribute_or(left, right)

    return node


# ============================================================
# 9. CNF CONVERSION PIPELINE
# ============================================================

def process_formula(expression):

    global skolem_counter

    skolem_counter = 0

    original = parse_expression(expression)

    stage1 = eliminate_implications(original)

    stage2 = make_nnf(stage1)

    stage3 = skolemize(stage2)

    stage4 = convert_to_cnf(stage3)

    return {
        "Input": show(original),
        "Implication Elimination": show(stage1),
        "NNF": show(stage2),
        "Skolemization": show(stage3),
        "Final CNF": show(stage4),
        "cnf_tree": stage4
    }


# ============================================================
# 10. CONVERT CNF INTO CLAUSES
# ============================================================

def extract_literals(node):

    if node[0] == "or":

        return (
            extract_literals(node[1])
            + extract_literals(node[2])
        )

    if node[0] == "not":

        atom = node[1]

        return [(False, atom[1], atom[2])]

    if node[0] == "pred":

        return [(True, node[1], node[2])]

    raise ValueError("Invalid CNF literal")


def cnf_to_clause_list(node):

    if node[0] == "and":

        return (
            cnf_to_clause_list(node[1])
            + cnf_to_clause_list(node[2])
        )

    return [extract_literals(node)]


def clause_to_text(clause):

    if not clause:
        return "[]"

    result = []

    for sign, predicate, arguments in clause:

        atom = (
            predicate
            + "("
            + ", ".join(arguments)
            + ")"
        )

        if not sign:
            atom = "¬" + atom

        result.append(atom)

    return " ∨ ".join(result)


# ============================================================
# 11. VARIABLE AND UNIFICATION FUNCTIONS
# ============================================================

def is_variable(term):

    # Variables are marked with ? after standardization.
    return term.startswith("?")


def apply_substitution(term, substitution):

    while term in substitution:

        new_term = substitution[term]

        if new_term == term:
            break

        term = new_term

    return term


def apply_to_literal(literal, substitution):

    sign, predicate, arguments = literal

    new_arguments = [
        apply_substitution(arg, substitution)
        for arg in arguments
    ]

    return (sign, predicate, new_arguments)


def unify_terms(a, b, substitution):

    a = apply_substitution(a, substitution)
    b = apply_substitution(b, substitution)

    if a == b:
        return True

    if is_variable(a):

        substitution[a] = b
        return True

    if is_variable(b):

        substitution[b] = a
        return True

    return False


def unify_literals(lit1, lit2):

    sign1, pred1, args1 = lit1
    sign2, pred2, args2 = lit2

    # Predicates must be opposite in sign
    if sign1 == sign2:
        return None

    if pred1 != pred2:
        return None

    if len(args1) != len(args2):
        return None

    substitution = {}

    for a, b in zip(args1, args2):

        if not unify_terms(a, b, substitution):
            return None

    return substitution


# ============================================================
# 12. STANDARDIZE VARIABLES APART
# ============================================================

def standardize_apart(clause, suffix):

    mapping = {}

    for sign, predicate, arguments in clause:

        for arg in arguments:

            if is_variable(arg):
                mapping[arg] = arg + suffix

    result = []

    for sign, predicate, arguments in clause:

        new_arguments = [
            mapping.get(arg, arg)
            for arg in arguments
        ]

        result.append(
            (sign, predicate, new_arguments)
        )

    return result


# ============================================================
# 13. CANONICAL CLAUSE
# ============================================================

def canonical_clause(clause):

    mapping = {}
    normalized = []

    for sign, predicate, arguments in clause:

        new_arguments = []

        for arg in arguments:

            if is_variable(arg):

                if arg not in mapping:

                    mapping[arg] = (
                        "?V" + str(len(mapping))
                    )

                new_arguments.append(mapping[arg])

            else:

                new_arguments.append(arg)

        normalized.append(
            (sign, predicate, tuple(new_arguments))
        )

    return tuple(sorted(normalized, key=str))


# ============================================================
# 14. RESOLUTION OPERATION
# ============================================================

def resolve_pair(clause1, clause2):

    resolvents = []

    for i, lit1 in enumerate(clause1):

        for j, lit2 in enumerate(clause2):

            substitution = unify_literals(lit1, lit2)

            if substitution is None:
                continue

            remaining = (
                clause1[:i]
                + clause1[i + 1:]
                + clause2[:j]
                + clause2[j + 1:]
            )

            new_clause = [
                apply_to_literal(lit, substitution)
                for lit in remaining
            ]

            # Remove duplicate literals
            unique = []

            for literal in new_clause:

                if literal not in unique:
                    unique.append(literal)

            resolvents.append(unique)

    return resolvents


# ============================================================
# 15. RESOLUTION PROOF WITH TREE RECORDING
# ============================================================

MAX_RESOLUTION_STEPS = 5000


def resolution_proof(kb_clauses, negated_goal_clauses):

    graph = nx.DiGraph()

    clauses = []
    known = set()

    node_counter = 0
    steps = 0

    # Add all initial clauses
    initial_clauses = (
        kb_clauses + negated_goal_clauses
    )

    for clause in initial_clauses:

        key = canonical_clause(clause)

        if key in known:
            continue

        known.add(key)

        node_id = "C" + str(node_counter)
        node_counter += 1

        clauses.append((node_id, clause))

        graph.add_node(
            node_id,
            label=clause_to_text(clause)
        )

        if len(clause) == 0:

            return True, graph, node_id, steps

    # Resolution search
    while steps < MAX_RESOLUTION_STEPS:

        new_items = []

        for (id1, clause1), (id2, clause2) in itertools.combinations(
            clauses, 2
        ):

            steps += 1

            c1 = standardize_apart(clause1, "_A")
            c2 = standardize_apart(clause2, "_B")

            resolvents = resolve_pair(c1, c2)

            for resolvent in resolvents:

                key = canonical_clause(resolvent)

                if key in known:
                    continue

                known.add(key)

                node_id = "C" + str(node_counter)
                node_counter += 1

                graph.add_node(
                    node_id,
                    label=clause_to_text(resolvent)
                )

                graph.add_edge(id1, node_id)
                graph.add_edge(id2, node_id)

                # Empty clause is derived
                if len(resolvent) == 0:

                    return True, graph, node_id, steps

                new_items.append((node_id, resolvent))

                if steps >= MAX_RESOLUTION_STEPS:
                    break

            if steps >= MAX_RESOLUTION_STEPS:
                break

        if not new_items:

            return False, graph, None, steps

        clauses.extend(new_items)

    return False, graph, None, steps


# ============================================================
# 16. PLOT RESOLUTION TREE
# ============================================================

def plot_resolution_tree(graph, empty_node):

    labels = nx.get_node_attributes(
        graph,
        "label"
    )

    # Use a hierarchical layout if Graphviz is installed.
    # Otherwise use NetworkX's built-in layout.
    try:

        positions = nx.nx_agraph.graphviz_layout(
            graph,
            prog="dot"
        )

    except Exception:

        positions = nx.spring_layout(
            graph,
            seed=42,
            k=2.0
        )

    plt.figure(figsize=(18, 12))

    node_colors = []

    for node in graph.nodes():

        if node == empty_node:
            node_colors.append("lightgreen")
        else:
            node_colors.append("lightblue")

    nx.draw_networkx_nodes(
        graph,
        positions,
        node_size=2800,
        node_color=node_colors,
        edgecolors="black"
    )

    nx.draw_networkx_edges(
        graph,
        positions,
        arrows=True,
        arrowsize=20,
        width=1.5
    )

    nx.draw_networkx_labels(
        graph,
        positions,
        labels,
        font_size=9
    )

    plt.title(
        "Resolution Proof Tree - Empty Clause Derived",
        fontsize=16
    )

    plt.axis("off")
    plt.tight_layout()
    plt.show()


# ============================================================
# 17. MAIN PROGRAM
# ============================================================

def main():

    print("=" * 65)
    print("       FIRST ORDER LOGIC RESOLUTION SYSTEM")
    print("=" * 65)

    print("\nOperator syntax:")
    print("forall x = ∀x")
    print("exists x = ∃x")
    print("not      = ¬")
    print("and      = ∧")
    print("or       = ∨")
    print("implies  = →")
    print("iff      = ↔")

    print("\nExample:")
    print("forall x (Human(x) implies Mortal(x))")

    # --------------------------------------------------------
    # STEP 1: GET ANY NUMBER OF KB STATEMENTS
    # --------------------------------------------------------

    while True:

        try:

            number = int(
                input("\nEnter number of KB statements: ")
            )

            if number < 0:
                print("Enter zero or a positive number.")
                continue

            break

        except ValueError:

            print("Enter a valid integer.")

    kb_inputs = []

    print("\nEnter the KB statements:")

    for i in range(number):

        while True:

            try:

                statement = input(
                    f"\nKB Statement {i + 1}: "
                )

                parse_expression(statement)

                kb_inputs.append(statement)

                break

            except (ValueError, IndexError) as error:

                print("Invalid statement:", error)

    # --------------------------------------------------------
    # STEP 2: GET GOAL
    # --------------------------------------------------------

    while True:

        try:

            goal_input = input(
                "\nEnter the goal statement: "
            )

            parse_expression(goal_input)

            break

        except (ValueError, IndexError) as error:

            print("Invalid goal:", error)

    # --------------------------------------------------------
    # STEP 3: ALL INPUTS COLLECTED
    # --------------------------------------------------------

    print("\n" + "=" * 65)
    print("ALL INPUTS COLLECTED SUCCESSFULLY")
    print("=" * 65)

    print("Total KB statements:", len(kb_inputs))

    # --------------------------------------------------------
    # STEP 4: CONVERT KB INTO CNF
    # --------------------------------------------------------

    kb_clauses = []

    for i, statement in enumerate(kb_inputs, start=1):

        print("\n" + "=" * 65)
        print("KNOWLEDGE BASE STATEMENT", i)
        print("=" * 65)

        result = process_formula(statement)

        print("\nOriginal FOL:")
        print(result["Input"])

        print("\nImplication Elimination:")
        print(result["Implication Elimination"])

        print("\nNegation Normal Form:")
        print(result["NNF"])

        print("\nSkolemization:")
        print(result["Skolemization"])

        print("\nFinal CNF:")
        print(result["Final CNF"])

        clauses = cnf_to_clause_list(
            result["cnf_tree"]
        )

        print("\nCNF Clauses:")

        for clause in clauses:

            print(clause_to_text(clause))

        kb_clauses.extend(clauses)

    # --------------------------------------------------------
    # STEP 5: CONVERT GOAL INTO CNF
    # --------------------------------------------------------

    print("\n" + "=" * 65)
    print("GOAL STATEMENT")
    print("=" * 65)

    goal_result = process_formula(goal_input)

    print("\nOriginal Goal:")
    print(goal_result["Input"])

    print("\nGoal CNF:")
    print(goal_result["Final CNF"])

    # --------------------------------------------------------
    # STEP 6: NEGATE GOAL
    # --------------------------------------------------------

    negated_goal = "not (" + goal_input + ")"

    negated_result = process_formula(negated_goal)

    negated_goal_clauses = cnf_to_clause_list(
        negated_result["cnf_tree"]
    )

    print("\n" + "=" * 65)
    print("NEGATED GOAL")
    print("=" * 65)

    print(negated_result["Final CNF"])

    print("\nNegated Goal Clauses:")

    for clause in negated_goal_clauses:

        print(clause_to_text(clause))

    # --------------------------------------------------------
    # STEP 7: APPLY RESOLUTION
    # --------------------------------------------------------

    print("\n" + "=" * 65)
    print("STARTING RESOLUTION PROOF")
    print("=" * 65)

    proved, resolution_graph, empty_node, steps = (
        resolution_proof(
            kb_clauses,
            negated_goal_clauses
        )
    )

    # --------------------------------------------------------
    # STEP 8: DISPLAY FINAL RESULT
    # --------------------------------------------------------

    print("\n" + "=" * 65)
    print("--- RESULT ---")
    print("=" * 65)

    if proved:

        print(
            "Resolution Tree Formed: Yes "
            "(Empty clause [] derived)"
        )

        print("Query Proven: TRUE")

        print("\nEmpty Clause:", "[]")

        print("\nPlotting Resolution Proof Tree...")

        # Plot only if empty clause is derived
        plot_resolution_tree(
            resolution_graph,
            empty_node
        )

    else:

        print(
            "Resolution Tree Formed: No "
            "(Empty clause [] not derived)"
        )

        print("Query Proven: FALSE")

        print("\nGraph will not be plotted.")

    print("\nResolution steps examined:", steps)


# ============================================================
# 18. EXECUTE PROGRAM
# ============================================================

if __name__ == "__main__":

    try:
        main()

    except Exception as error:

        print("\nProgram Error:", error)


       FIRST ORDER LOGIC RESOLUTION SYSTEM

Operator syntax:
forall x = ∀x
exists x = ∃x
not      = ¬
and      = ∧
or       = ∨
implies  = →
iff      = ↔

Example:
forall x (Human(x) implies Mortal(x))

Enter number of KB statements: 5

Enter the KB statements:

KB Statement 1: forall x (Science(x) implies (Physics(x) or Mathematics(x)))

KB Statement 2: forall x (Mathematics(x) implies Statistics(x))

KB Statement 3: Science(Alice)

KB Statement 4: Physics(Bob)

KB Statement 5: forall x (Statistics(x) implies DataAnalysis(x))

Enter the goal statement: DataAnalysis(Alice)

ALL INPUTS COLLECTED SUCCESSFULLY
Total KB statements: 5

KNOWLEDGE BASE STATEMENT 1

Original FOL:
∀x ((Science(x) → (Physics(x) ∨ Mathematics(x))))

Implication Elimination:
∀x ((¬Science(x) ∨ (Physics(x) ∨ Mathematics(x))))

Negation Normal Form:
∀x ((¬Science(x) ∨ (Physics(x) ∨ Mathematics(x))))

Skolemization:
(¬Science(x) ∨ (Physics(x) ∨ Mathematics(x)))

Final CNF:
(¬Science(x) ∨ (Physics(x) ∨ Mathematics(x))